In [1]:

# EfficientNet-B0 — Baseline Training 


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from sklearn.model_selection import train_test_split
import numpy as np
import os


# Device & performance

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224, 224), antialias=True),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")

# Stratified split 85/10/5
targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for c in np.unique(targets):
    idxs = np.where(targets == c)[0]
    tr, tmp = train_test_split(idxs, test_size=0.15,
                               random_state=42, shuffle=True)
    va, te = train_test_split(tmp, test_size=1/3,
                               random_state=42, shuffle=True)
    train_idx.extend(tr); val_idx.extend(va); test_idx.extend(te)

train_dataset, val_dataset, test_dataset = (
    Subset(dataset, train_idx),
    Subset(dataset, val_idx),
    Subset(dataset, test_idx)
)

batch_size = 128
num_workers = min(8, os.cpu_count())
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True)
print("DataLoaders ready")


# Model

weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

num_classes = len(dataset.classes)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

model = model.to(device, memory_format=torch.channels_last)

if hasattr(torch, "compile"):
    model = torch.compile(model, mode="max-autotune")


# Loss, Optimizer, Scheduler

criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=1e-4, nesterov=True)
epochs = 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


# Evaluation function

def evaluate_model(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return 100 * correct / total


# Training loop (no early stopping)

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=20):
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=(device.type=="cuda")):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_acc = 100 * correct / total
        val_acc = evaluate_model(model, val_loader, device)

        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Train Loss: {running_loss/total:.4f} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

        scheduler.step()


# Run Training

train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=20)


# Save final weights

torch.save(model.state_dict(), "efficientnet_b0_weights.pth")
print("Saved EfficientNet-B0 weights")


# Final test accuracy

test_acc = evaluate_model(model, test_loader, device)
print(f"Final Test Accuracy: {test_acc:.2f}%")


Using device: cuda
Total images: 100000, Classes: 200
✅ DataLoaders ready


AUTOTUNE convolution(128x3x224x224, 32x3x3x3)
  triton_convolution_4 0.4700 ms 100.0%
  convolution 0.4915 ms 95.6%
  triton_convolution_3 0.5181 ms 90.7%
  triton_convolution_2 0.5509 ms 85.3%
  triton_convolution_0 0.6420 ms 73.2%
  triton_convolution_5 0.7240 ms 64.9%
  triton_convolution_1 1.0066 ms 46.7%
SingleProcess AUTOTUNE takes 60.6123 seconds
AUTOTUNE addmm(128x8, 128x32, 32x8)
  triton_mm_11 0.0082 ms 100.0%
  triton_mm_12 0.0082 ms 100.0%
  triton_mm_13 0.0082 ms 100.0%
  triton_mm_6 0.0092 ms 88.9%
  triton_mm_7 0.0092 ms 88.9%
  triton_mm_8 0.0092 ms 88.9%
  triton_mm_9 0.0092 ms 88.9%
  triton_mm_10 0.0092 ms 88.9%
  triton_mm_14 0.0092 ms 88.9%
  triton_mm_15 0.0092 ms 88.9%
SingleProcess AUTOTUNE takes 9.6414 seconds
AUTOTUNE addmm(128x32, 128x8, 8x32)
  triton_mm_22 0.0072 ms 100.0%
  triton_mm_23 0.0072 ms 100.0%
  triton_mm_25 0.0072 ms 100.0%
  triton_mm_27 0.0072 ms 100.0%
  triton_mm_17 0.0082 ms 87.5%
  triton_mm_18 0.0082 ms 87.5%
  triton_mm_19 0.0082 ms 87.5

Epoch [1/20] | Train Loss: 2.6579 | Train Acc: 50.38% | Val Acc: 54.00%
Epoch [2/20] | Train Loss: 2.1481 | Train Acc: 63.15% | Val Acc: 58.94%
Epoch [3/20] | Train Loss: 1.9554 | Train Acc: 68.47% | Val Acc: 60.40%
Epoch [4/20] | Train Loss: 1.8330 | Train Acc: 71.94% | Val Acc: 61.46%
Epoch [5/20] | Train Loss: 1.7347 | Train Acc: 74.95% | Val Acc: 63.50%
Epoch [6/20] | Train Loss: 1.6499 | Train Acc: 77.53% | Val Acc: 60.79%
Epoch [7/20] | Train Loss: 1.5735 | Train Acc: 79.73% | Val Acc: 54.56%
Epoch [8/20] | Train Loss: 1.4923 | Train Acc: 82.43% | Val Acc: 66.12%
Epoch [9/20] | Train Loss: 1.4099 | Train Acc: 85.16% | Val Acc: 63.91%
Epoch [10/20] | Train Loss: 1.3345 | Train Acc: 87.80% | Val Acc: 68.77%
Epoch [11/20] | Train Loss: 1.2505 | Train Acc: 90.78% | Val Acc: 68.68%
Epoch [12/20] | Train Loss: 1.1872 | Train Acc: 93.01% | Val Acc: 70.68%
Epoch [13/20] | Train Loss: 1.1262 | Train Acc: 95.17% | Val Acc: 71.59%
Epoch [14/20] | Train Loss: 1.0776 | Train Acc: 96.76% | Val